# **FICORO_GNSS**
### An open-source Python software package for filtering, combining and rotating GNSS velocity fields
### Nicolás Castro-Perdomo, 2024
#### Indiana University, Bloomington, US
---

### Define input and output parameters

In [ ]:
"""Master script to filter and combine GPS velocity fields"""

# Import libraries and define paths and parameters
import os
import subprocess
import datetime
import shutil
import sys
import glob
import pandas as pd
import numpy as np
import pygmt

# Input parameters
input_path = "raw_input"
output_path = "raw_input_column_formatted"
input_file_ext = "raw"
output_file_ext = "vel"
scripts_path = "scripts"
results_path = "results"
igb_nocomb_path = "igb14_no_comb"
igb_nocomb_subpath = "igb14"
log_path = "logs"

lognorm_script_path = os.path.join(scripts_path, "lognorm_filter.py")
coherence_script_path = os.path.join(scripts_path, "coherence_filter.py")
combination_script_path = os.path.join(scripts_path, "combine_vel.py")
plot_maps_filter_path = os.path.join(scripts_path, "plot_maps_filtering.py")
coherence_results_path = os.path.join(results_path, "output_coherence_analysis")
input_files_rot_path = os.path.join(results_path, "input_files_rotation")
ITRF14_vel_path = os.path.join(results_path, "rotation_steps")
lnk_file = "velrot.lnk"
lnk_folder_path = "./rotation_lnk_file"
lnk_file_path = os.path.join(lnk_folder_path, lnk_file)
reference_vel = "serpelloni_2022.vel"
reference_vel_path = os.path.join(input_files_rot_path, reference_vel)
temp_combined_folder_path = os.path.join(results_path, "combined_velocities")
combined_folder_path = os.path.join(temp_combined_folder_path, "manual_filter")
plot_rotated_script_path = os.path.join(scripts_path, "plot_rotated_vels.py")
# Output folder for combined horizontal velocity fields with scaled uncertainties
output_folder_scaled_results = os.path.join(results_path, "combined_velocities_scaled_uncertainties")  

print("------------------------------------------------------------------------------------")
print(f"                   FICORO_GNSS executed on: {datetime.datetime.now().strftime('%Y-%m-%d, %H:%M:%S')}               ")
print("------------------------------------------------------------------------------------")
print(" Description: Python tools to filter, rotate and combine GNSS velocity fields       ")
print(" Default parameters:                                                                ")
print(f" - Input path including raw velocity files= {input_path}/                          ")
print(f" - Output path including combined horizontal velocities= {output_folder_scaled_results}/             ")
print(f" - Reference velocity field for alignment to ITRF14= {reference_vel}               ")
print(" - Headers: Lon Lat E.vel N.vel E.adj N.adj E.sig N.sig Corr U.vel U.adj U.sig Sta    ")
print("------------------------------------------------------------------------------------")
print()

---
### Load input velocity fields

In [ ]:
# Column formatting routine
print("------------------------------------------------------------------------------------")
print("                               Checking input files                                 ")
print("------------------------------------------------------------------------------------")
print()

# Check if the input path exists
if not os.path.exists(input_path):
    print(f"Input path '{input_path}' does not exist. Terminating.")
    exit(1)

# Check if the input path is empty
if not os.listdir(input_path):
    print("Input folder is empty. Terminating.")
    exit(1)

# Check if the output path exists
if os.path.exists(output_path):
    # Remove the files inside the output path
    for file in os.listdir(output_path):
        file_path = os.path.join(output_path, file)
        if os.path.isfile(file_path):
            os.remove(file_path)
else:
    # Create the output path if it doesn't exist
    os.makedirs(output_path)

# Loop through each input file in the input path
for input_file in os.listdir(input_path):
    if input_file.endswith("." + input_file_ext):
        base = os.path.splitext(input_file)[0]
        print(f"Translating {base}.{input_file_ext} into {base}.{output_file_ext}")
        with open(os.path.join(input_path, input_file), "r") as input_file:
            lines = input_file.readlines()
            lines = [line.strip().split() for line in lines]
            header = "Lon Lat E.vel N.vel E.adj N.adj E.sig N.sig Corr U.vel U.adj U.sig Stat"
            formatted_lines = [header] + [" ".join(line) for line in lines]
            output_file_path = os.path.join(output_path, f"{base}.{output_file_ext}")
            with open(output_file_path, "w") as output_file:
                output_file.write("\n".join(formatted_lines))

## **Step 1: Filter out stations affected by postseismic transients**

In [ ]:
import pandas as pd
import numpy as np
import os
import shutil  # For file operations like move
import warnings

# Ignore future warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

print("------------------------------------------------------------------------------------")
print("            Filtering GNSS stations affected by postseismic transients                ")
print("------------------------------------------------------------------------------------")
print()

# Define paths
input_file_path = os.path.join(output_path, "ergintav_2023.vel")  # Path to the input velocity file
temp_output_file_path = os.path.join(log_path, "ergintav_2023_filtered_temp.vel") # Temporary file path for filtered data
log_file_path = os.path.join(log_path, "ergintav_2023_removed_postseismic.log")  # Log file path for removed stations

# Ensure the existence of output directories
os.makedirs(os.path.dirname(temp_output_file_path), exist_ok=True)

def haversine(lon1, lat1, lon2, lat2):
    # Convert to radians
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c  # Earth's radius in kilometers

def filter_stations(file_path, temp_output_path, log_path, epicenters, distance_threshold):
    # Load velocity data
    df = pd.read_csv(file_path, sep=r"\s+", header=0)
    original_size = len(df)

    # For logging removed stations
    removed_stations = pd.DataFrame(columns=df.columns)

    # Filtering
    for epicenter in epicenters:
        lon, lat = epicenter
        distances = np.vectorize(haversine)(df['Lon'], df['Lat'], lon, lat)
        within_threshold = distances <= distance_threshold
        removed_stations = pd.concat([removed_stations, df[within_threshold]], ignore_index=True)
        df = df[~within_threshold]

    # Save filtered data to a temporary file and log
    df.to_csv(temp_output_path, sep=' ', index=False)
    removed_stations.to_csv(log_path, sep=' ', index=False)

    # Replace original file with the filtered data
    shutil.move(temp_output_path, file_path)

    # Summary
    final_size = len(df)
    removed_count = original_size - final_size
    percentage_removed = (removed_count / original_size) * 100
    print(f"Processed {os.path.basename(file_path)}:\nOriginal size={original_size}, Final size={final_size}, Removed={removed_count}, Percentage removed={percentage_removed:.2f}%")
    print(f"Log saved to {log_path}\nCleaned data saved to {file_path}")

epicenters = [(29.86, 40.75), (31.16, 40.76), (22, 40)]  # Epicenter coordinates (lon, lat) for Izmit and Duzce earthquakes and Halkidiki Peninsula region. See Ergintav et al., 2023 for details
distance_threshold = 150  # km

filter_stations(input_file_path, temp_output_file_path, log_file_path, epicenters, distance_threshold)

---
## **Steps 2 and 3: Filter stations by velocity uncertainties and spatial coherence**

In [ ]:
import json
import tempfile

# Running GNSS velocity cleaning/filtering scripts
print() 
print("------------------------------------------------------------------------------------")
print("                          Cleaning GNSS velocity fields                             ")
print("------------------------------------------------------------------------------------")
print()                

###########################################################################################
#########################  Execute lognormal filter script ###############################
###########################################################################################

if os.path.exists(lognorm_script_path):
    input_lognorm_path = "./raw_input_column_formatted"
    output_lognorm_path = "./results/output_lognorm_99_filtered"
    excluded_lognorm_path = "./results/sites_excluded_lognorm_99"
    figure_folder_path = "./results/figures"
    
    subprocess.run(["python", lognorm_script_path, input_lognorm_path, output_lognorm_path, excluded_lognorm_path, figure_folder_path])
else:
    print(f"Error: {lognorm_script_path} not found. Terminating")
    exit(1)

print()     

###########################################################################################
#####  Coherence filter with optional user-defined geographic-based stringency levels #####
###########################################################################################

# Enable or disable geographic stringency feature 
geo_strict_enabled = True  # Change to False to disable this feature

# Define geographic regions with variable stringency levels. The default sigma value is 2.
regions = [
    {'min_lon': 117.93, 'max_lon': 123.16, 'min_lat': 21.85, 'max_lat': 25.75, 'sigma': 3}, # Taiwan
    {'min_lon': 34.35, 'max_lon': 36.87, 'min_lat': 29.54, 'max_lat': 36.24, 'sigma': 3}, # Dead Sea Fault area
    {'min_lon': 29, 'max_lon': 34, 'min_lat': 40, 'max_lat': 40.15, 'sigma': 3}, # Izmit and Ismetpasa creeping faults  
    # Add more regions as needed...
]

# Path to the folder containing output data from the lognormal filter
input_folder_path = output_lognorm_path

# Define the special case file to be skipped in the coherence analysis
#special_case_file = 'ozarpaci' # Velocities from Ozarpaci et al., 2020, GJI, along Izmit and Ismetpasa faults aren't filtered to preserve afterslip creeping signal

# Check if the special case file exists
#if special_case_file not in os.listdir(input_folder_path):
#    print(f"Error: {special_case_file} not found in {input_folder_path}. Terminating")
#    exit(1)

# Define the pattern for special case files
special_case_file = 'ozarpaci' # Velocities from Ozarpaci et al., 2020, GJI, along Izmit and Ismetpasa faults aren't filtered to preserve afterslip creeping signal

# Check if any file contains the pattern in its name
special_case_found = False
for filename in os.listdir(input_folder_path):
    if special_case_file in filename:
        special_case_found = True
        break

# If no file with the special case pattern is found
if not special_case_found:
    print(f"Error: No file containing '{special_case_file}' found in {input_folder_path}. Terminating.")
    exit(1)


# Check if the coherence filter script exists
if not os.path.exists(coherence_script_path):
    print(f"Error: {coherence_script_path} not found. Terminating")
    exit(1)

# Execute the coherence filter script

if geo_strict_enabled:
    # Create a temporary file to store the regions if the feature is enabled
    with tempfile.NamedTemporaryFile(delete=False, mode='w', suffix='.json') as tmpfile:
        regions_json_path = tmpfile.name
        json.dump(regions, tmpfile)
    
    command = [
        "python", coherence_script_path, input_folder_path,
        '--geo_strict', '--regions_json', regions_json_path, '--special_case_file', special_case_file
    ]
else:
    command = [
        "python", coherence_script_path, input_folder_path,
        '--special_case_file', special_case_file
    ]

# Execute the coherence filter script
if os.path.exists(coherence_script_path):
    subprocess.run(command)
    if geo_strict_enabled:
        # Remove the temporary JSON file after the analysis
        os.remove(regions_json_path)
else:
    print(f"Error: {coherence_script_path} not found. Terminating")
    exit(1)

#### Applying a Lower Bound to Velocity Uncertainties for Certain Input Velocity Fields Reporting Unrealistically Low Values
- Some input velocity fields report unrealistically low velocity uncertainties (e.g., < 0.1 mm/yr), likely due to the use of formal uncertainties derived from fitting a trajectory model to GNSS timeseries, considering only white noise.
- Such low uncertainties are problematic, as they can introduce numerical instabilities when performing weighted least-squares inversions for strain rates.
- To address this, I identified velocity fields reporting formal uncertainties and imposed a lower bound of 0.5 mm/yr on them to ensure stability in the inversion process.

In [ ]:
import os
import pandas as pd

print() 
print("------------------------------------------------------------------------------------")
print("                    Applying a lower bound to velocity uncertainties                ")           
print("------------------------------------------------------------------------------------")
print()       

# List of filenames without extension
file_names = ['bahrouni_2020']  # velocity files reporting some unrealistically low uncertainties,

# Check if files exist and apply a lower bound of 0.5 mm/yr to their velocity uncertainties
for file_name in file_names:
    file_path = os.path.join(coherence_results_path, file_name + '.csv')  # Automatically append '.csv'
    
    if os.path.exists(file_path):
        # Read the file, skipping the first row and assigning column names
        df = pd.read_csv(file_path, sep=' ', skiprows=1, header=None)
        df.columns = ['Lon', 'Lat', 'E.vel', 'N.vel', 'E.adj', 'N.adj', 'E.sig', 'N.sig', 'Corr', 'U.vel', 'U.adj', 'U.sig', 'Stat']
        
        # Apply lower bound to E.sig and N.sig (0.5 mm/yr)
        df['E.sig'] = df['E.sig'].apply(lambda x: max(x, 0.5))
        df['N.sig'] = df['N.sig'].apply(lambda x: max(x, 0.5))
        
        # Save the modified file back to the original location
        df.to_csv(file_path, sep=' ', index=False, header=False)
        print(f"Applied a lower bound of 0.5mm/yr to velocity uncertainties in {file_name}.csv")
    else:
        print(f"Warning: File {file_name}.csv does not exist in the folder {coherence_results_path}.")

---
#### Plot filtered GNSS velocities and outliers for each input velocity field

In [ ]:
# User flag: set to True to plot, False to skip plotting
do_plots = False  # Change to True to enable plotting

if do_plots:
    print()
    print("------------------------------------------------------------------------------------")
    print("                          Plotting GNSS velocity fields                             ")
    print("------------------------------------------------------------------------------------")
    print()

    # Execute plotting script
    if os.path.exists(plot_maps_filter_path):
        subprocess.run(["python", plot_maps_filter_path])
    else:
        print(f"Error: {plot_maps_filter_path} not found. Terminating")
        exit(1)
else:
    print("Plotting skipped because do_plots is False.")

---
## Step 4: Align input velocity fields to ITRF14 

In [ ]:
# Column formatting routine for cvframe/velrot compatibility

# Check if the input folder is empty
if not os.listdir(coherence_results_path):
    print("Input folder is empty. Terminating.")
    exit(1)

# Check if the folder containing input files for rotation exists
if os.path.exists(input_files_rot_path):
    # Remove all contents of the input_files_rot_path folder
    for file in os.listdir(input_files_rot_path):
        file_path = os.path.join(input_files_rot_path, file)
        if os.path.isfile(file_path):
            os.remove(file_path)
else:
    # Create the input_files_rot_path folder if it doesn't exist
    os.makedirs(input_files_rot_path)

# Loop through each csv file in the input folder
for file in os.listdir(coherence_results_path):
    if file.endswith(".csv"):
        input_file_path = os.path.join(coherence_results_path, file)
        output_file_path = os.path.join(input_files_rot_path, os.path.splitext(file)[0] + ".vel")
        with open(input_file_path, "r") as input_file:
            lines = input_file.readlines()
            lines = [line.strip() for line in lines[1:]]
            formatted_lines = [" " + line for line in lines]
            with open(output_file_path, "w") as output_file:
                output_file.write("\n".join(formatted_lines))

# Aligning velocity fields to ITRF14
print()
print("--------------------------------------------------------------")
print("              Aligning velocity fields to ITRF14              ")
print("--------------------------------------------------------------")
print()

# Check if the input folder is empty
if not os.listdir(input_files_rot_path):
    print("Input folder is empty. Terminating.")
    exit(1)

# Check if the folder containing ITRF14-rotated velocities exists
if os.path.exists(ITRF14_vel_path):
    # Remove all contents of the ITRF14_vel_path folder
    for file in os.listdir(ITRF14_vel_path):
        file_path = os.path.join(ITRF14_vel_path, file)
        if os.path.isfile(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
else:
    # Create the ITRF14_vel_path folder if it doesn't exist
    os.makedirs(ITRF14_vel_path)

if not os.path.exists(lnk_file_path):
    print(f"Creating {lnk_file}...")
    os.makedirs(lnk_folder_path, exist_ok=True)
    with open(lnk_file_path, "w") as f:
        f.write(" " + "eq_dist 1000\n")

if not os.path.exists(reference_vel_path):
    print(f"Error: The reference velocity file {reference_vel_path} does not exist.")
    exit(1)
            
print("Formatting files before alignment to ITRF14 completed")

top_pattern = "SYSTEM 1 Velocities transformed to SYSTEM 2"
bot_pattern = "SYSTEM 2 Velocities except those in SYSTEM 1"

# Assign the link file name as a string
lnk_file_str = "velrot.lnk"

# Jupyter-safe path handling
script_dir = os.getcwd()

pyvelrot_path = os.path.join(script_dir, "scripts", "pyvelrot.py")

# Optional safety check
if not os.path.exists(pyvelrot_path):
    raise FileNotFoundError(
        f"pyvelrot.py not found at: {pyvelrot_path}"
    )

ref_base = os.path.splitext(os.path.basename(reference_vel))[0]
ref_name = os.path.basename(reference_vel_path)

for vel_file in os.listdir(input_files_rot_path):
    if vel_file.endswith(".vel"):
        base = os.path.splitext(vel_file)[0]
        folder_path = os.path.join(ITRF14_vel_path, base)
        os.makedirs(folder_path, exist_ok=True)

        shutil.copy(os.path.join(input_files_rot_path, vel_file), folder_path)
        shutil.copy(reference_vel_path, folder_path)
        shutil.copy(lnk_file_path, folder_path)

        if base == ref_base:
            os.rename(
                os.path.join(folder_path, ref_name),
                os.path.join(folder_path, f"{base}_igb14.vel")
            )
        else:
            cmd = [
                sys.executable, pyvelrot_path,
                f"{base}.vel", "NONE", reference_vel,
                "NONE", f"{base}_igb14.tmp", "NONE", lnk_file_str, "0", "TR"
            ]
            log_file_path = os.path.join(folder_path, f"{base}_align.log")

            with open(log_file_path, "w") as log_file:
                subprocess.run(
                    cmd,
                    cwd=folder_path,
                    stdout=log_file,
                    stderr=subprocess.STDOUT,
                    check=True
                )

            rotated_tmp_file = os.path.join(folder_path, f"{base}_igb14.tmp")
            rotated_vel_file = os.path.join(folder_path, f"{base}_igb14.vel")

            with open(rotated_tmp_file, "r") as tmp_file:
                tmp_lines = tmp_file.readlines()

            start_idx = next(i for i, line in enumerate(tmp_lines) if top_pattern in line)
            end_idx = next(i for i, line in enumerate(tmp_lines) if bot_pattern in line)
            extracted_lines = tmp_lines[start_idx + 1:end_idx - 1]
            extracted_lines = extracted_lines[2:]

            cleaned_lines = [line.replace("*", "").rstrip() for line in extracted_lines]

            with open(rotated_vel_file, "w") as vel_file_out:
                vel_file_out.write("\n".join(cleaned_lines))

# Copying rotated velocity files to igb14_no_comb/igb14

# Create igb14_no_comb folder if it doesn't exist
igb_nocomb_parent_folder = os.path.join(results_path, igb_nocomb_path)
if not os.path.exists(igb_nocomb_parent_folder):
    os.makedirs(igb_nocomb_parent_folder)

# Create the igb14 if it doesn't exist
igb_nocomb_subfolder = os.path.join(igb_nocomb_parent_folder, igb_nocomb_subpath)
if not os.path.exists(igb_nocomb_subfolder):
    os.makedirs(igb_nocomb_subfolder)

# Loop through each folder in the ITRF14_vel_path
for folder in os.listdir(ITRF14_vel_path):
    folder_path = os.path.join(ITRF14_vel_path, folder)
    if os.path.isdir(folder_path):
        folder_name = os.path.basename(folder_path)

        # Check if the folder contains the desired file
        igb14_vel_file = os.path.join(folder_path, f"{folder_name}_igb14.vel")
        if os.path.exists(igb14_vel_file):
            # Copy the file to the igb_nocomb_subpath
            shutil.copy(igb14_vel_file, igb_nocomb_subfolder)

print("Alignment to igb14 completed. Rotated files copied to igb14_no_comb/igb14")

---
## Step 5: Rotate velocity fields using Euler poles

In [ ]:
# Rotating velocity fields using Euler poles
print()
print("--------------------------------------------------------------")
print("          Rotating velocity fields using Euler poles          ")
print("--------------------------------------------------------------")
print()

# Set the destination folder paths
arab_dest_folder = os.path.join(results_path, igb_nocomb_path, "arab")
eura_dest_folder = os.path.join(results_path, igb_nocomb_path, "eura")
nubi_dest_folder = os.path.join(results_path, igb_nocomb_path, "nubi")
sina_dest_folder = os.path.join(results_path, igb_nocomb_path, "sina")
anat_dest_folder = os.path.join(results_path, igb_nocomb_path, "anat")
indi_dest_folder = os.path.join(results_path, igb_nocomb_path, "indi")
igb14_dest_folder = os.path.join(results_path, igb_nocomb_path, igb_nocomb_subpath)
#myan_dest_folder = os.path.join(results_path, igb_nocomb_path, "myan")
amur_dest_folder = os.path.join(results_path, igb_nocomb_path, "amur")
yang_dest_folder = os.path.join(results_path, igb_nocomb_path, "yang")
aege_dest_folder = os.path.join(results_path, igb_nocomb_path, "aege")
tbet_dest_folder = os.path.join(results_path, igb_nocomb_path, "tbet")

# Create the destination folders
os.makedirs(arab_dest_folder, exist_ok=True)
os.makedirs(eura_dest_folder, exist_ok=True)
os.makedirs(nubi_dest_folder, exist_ok=True)
os.makedirs(sina_dest_folder, exist_ok=True)
os.makedirs(anat_dest_folder, exist_ok=True)
os.makedirs(indi_dest_folder, exist_ok=True)
#os.makedirs(myan_dest_folder, exist_ok=True)
os.makedirs(amur_dest_folder, exist_ok=True)
os.makedirs(yang_dest_folder, exist_ok=True)
os.makedirs(aege_dest_folder, exist_ok=True)
os.makedirs(tbet_dest_folder, exist_ok=True)

# Resolve absolute path to pycvframe.py
script_dir = os.getcwd()
pycvframe_path = os.path.join(script_dir, "scripts", "pycvframe.py")

if not os.path.exists(pycvframe_path):
    raise FileNotFoundError(f"pycvframe.py not found at: {pycvframe_path}")

# Define the Euler pole parameters for each rotation.
# Rotation pole cartesian components [wx, wy, wz] should be expressed in DEG/My
euler_poles = {
    "arab": "0.32840 -0.03504 0.40682", # Arabia Euler Pole by Viltres et al. (2022)
    "sina": "0.24250 -0.04797 0.34939", # Sinai Euler Pole by Hamiel et al. (2021)
    "anat": "1.008722 0.543127 1.020384", # Anatolia Euler Pole by Ergintav et al. (2023)
    "nubi": "0.02740 -0.1704 0.2037", # Nubia Euler Pole, ITRF14 by Altamimi et al. (2016) - Table S1
    "eura": "-0.0235 -0.1476 0.2140", # Eurasia Euler Pole, ITRF14 by Altamimi et al. (2016) - Table S1
    "indi": "0.3205 -0.0014 0.4038", # India Euler Pole by Altamimi et al. (2016) - Table S1
    #"amur": "-0.0364 -0.1532 0.2325", # Amur Euler Pole by Altamimi et al. (2023) - Table S1
    "amur": "-0.04800 -0.10714 0.27414", # Amur Euler Pole derived in Castro-Perdomo et al,. (2025).
    "yang": "-0.0578 -0.1272 0.2933", # Yangtze Euler Pole by K. Vardic et al. (2022) - Table 3
    "aege": "0.050898 0.147426 0.158303", # West Aegean Euler Pole by Ergintav et al. (2023)
    #"myan": "0.10292  0.71094  0.58459", # Myanmar Euler Pole by Lindsey et al. (2023)
    #"tbet": "0.13911 -0.61729  0.03546" # Central Tibet Euler pole derived in Castro-Perdomo et al,. (2025). Tibet is deforming internally, but the results are very illustrative.
    "tbet": "0.12983 -0.71360  -0.02336" # Central Tibet Euler pole derived in Castro-Perdomo et al,. (2025).
}

# Loop through each .vel file in the IGB14 folder
for velfile in os.listdir(igb14_dest_folder):
    if velfile.endswith(".vel"):
        base = os.path.splitext(velfile)[0]
        new_base = base.replace("_igb14", "")

        # Loop through each reference frame and rotate the .vel file
        for ref_frame, euler_params in euler_poles.items():
            output_vel_name = f"{new_base}_{ref_frame}.vel"

            # Skip the IGB14 reference frame itself
            if ref_frame == "igb14":
                continue

            # Run cvframe command to rotate the .vel file
            cmd = [
                sys.executable, pycvframe_path, velfile, output_vel_name, "ITRF14", euler_params
            ]
            result = subprocess.run(
                cmd, cwd=igb14_dest_folder,
                stdout=subprocess.PIPE, stderr=subprocess.PIPE
            )

            # Check if the output file was created before moving
            output_vel_path = os.path.join(igb14_dest_folder, output_vel_name)
            if not os.path.exists(output_vel_path):
                print(f"WARNING: pycvframe failed for {velfile} -> {output_vel_name}")
                print(f"  stdout: {result.stdout.decode().strip()}")
                print(f"  stderr: {result.stderr.decode().strip()}")
                continue

            # Move the rotated file to the destination folder
            dest_folder = (
                arab_dest_folder if ref_frame == "arab" else
                eura_dest_folder if ref_frame == "eura" else
                nubi_dest_folder if ref_frame == "nubi" else
                sina_dest_folder if ref_frame == "sina" else
                indi_dest_folder if ref_frame == "indi" else
                amur_dest_folder if ref_frame == "amur" else
                yang_dest_folder if ref_frame == "yang" else
                #myan_dest_folder if ref_frame == "myan" else
                aege_dest_folder if ref_frame == "aege" else
                tbet_dest_folder if ref_frame == "tbet" else
                anat_dest_folder
            )
            shutil.move(
                os.path.join(igb14_dest_folder, output_vel_name),
                os.path.join(dest_folder, output_vel_name)
            )

print("Rotation using Euler poles completed")

---
## Steps 6 and 7: Combination and filtering by velocity magnitude and azimuthal direction

In [ ]:
import concurrent.futures
import psutil

# --------------------------------------------------------------
# Helper functions
# --------------------------------------------------------------
def get_available_memory_gb():
    """Return available system memory in GB."""
    return psutil.virtual_memory().available / (1024 ** 3)

def combine_velocities(ref_frame):
    """Run the combination script for one reference frame."""
    print(f"Combining GPS velocities in {ref_frame}-fixed reference frame ...")
    ref_frame_folder = os.path.join(results_path, igb_nocomb_path, ref_frame)

    subprocess.run(
        ["python", combination_script_path, ref_frame_folder, temp_combined_folder_path],
        check=True
    )

# --------------------------------------------------------------
# Combining rotated velocity fields
# --------------------------------------------------------------
print("--------------------------------------------------------------")
print("               Combining rotated velocity fields              ")
print("--------------------------------------------------------------")
print()

if os.path.exists(combination_script_path):

    available_gb = get_available_memory_gb()
    print(f"Available memory: {available_gb:.2f} GB")

    frames = list(euler_poles.keys()) + ["igb14"]

    # Run sequentially if memory is low
    if available_gb < 3:

        print("Low memory detected; running sequentially.")
        print("--------------------------------------------------------------")
        print()

        for ref_frame in frames:
            combine_velocities(ref_frame)

    # Otherwise run in parallel, but do not exceed CPU count or number of frames
    else:

        total_workers = min(os.cpu_count() or 1, len(frames))

        print(f"Running combination script in parallel using {total_workers} workers")
        print("--------------------------------------------------------------")
        print()

        with concurrent.futures.ThreadPoolExecutor(max_workers=total_workers) as executor:
            list(executor.map(combine_velocities, frames))

else:
    print(f"Error: {combination_script_path} not found. Terminating")
    sys.exit(1)

### **Number of independent velocity estimates at each GNSS station**

In [ ]:
# Here I plot the number of independent velocity estimates at each GNSS station in the study area.
# For this purpose, I count the number of sites within a radius of 0.01 degrees (1.11 km) from each station.
# This step was done previously in the script 'combine_vel.py' and the results were stored in the file 
# 'results/combined_velocities/statistics/site_statistics.csv'. The columns of this file are: 'Lon', 'Lat', 
# 'Stat', 'Num', where 'Num' is the number of independent velocity estimates at each station.

import pygmt
import numpy as np
import pandas as pd

def plot_number_of_estimates(input_file):

    # Create a new figure
    fig = pygmt.Figure()

    # Set the region of the map
    region = [-20, 125, 5, 60]

    # Set the region and projection of the map
    fig.basemap(region=region, projection='M20c', frame='af')

    # Create a custom color palette for the relief shading
    pygmt.makecpt(cmap="gray95,gray90,gray85", series=[-10000, 10000, 100])

    # Add shaded topography with transparency
    fig.grdimage(grid="@earth_relief_03m", cmap=True, shading=True, transparency=20) # nan_transparent=True results in error in the latest GMT version

    # Add coastlines
    fig.coast(water='white', borders="1/0.1p,gray90", shorelines="0.1p,black", area_thresh=4000, resolution='h')

    # Read the CSV file
    df = pd.read_csv(input_file)

    # Cap the 'Num' values at 15
    df['Num'] = df['Num'].clip(upper=15)
    
    # Apply the log transformation
    df['Log_Num'] = np.log10(df['Num'])

    # Define the minimum and maximum for the colormap based on the transformed 'Log_Num' column
    min_val = df['Log_Num'].min()
    max_val = df['Log_Num'].max()

    # Adjust the colormap range based on the data
    pygmt.makecpt(cmap='turbo', series=[min_val, max_val, 0.05], continuous=False, background='o', log=True) # reverse=True
    
    # Create triangles with fixed size (e.g., 0.1 cm) and colors based on 'Num'
    triangles = [[lon, lat, num, 0.1] for lon, lat, num in zip(df['Lon'], df['Lat'], df['Num'])]
    
    # Plot triangles with color based on 'Num' and fixed size
    fig.plot(
        data=triangles,
        style='tc',
        cmap=True,
        pen='0.01p,black',
    )
    
    # Add scale bar
    with pygmt.config(FONT_ANNOT_PRIMARY='8p', FONT_LABEL='8p'):
        fig.basemap(map_scale="JBR+o-9c/-0.8c+c0+w1000k+f+lkm")

    # Add a colorbar to the plot
    fig.colorbar(position="JMR+o0.5c/0c+w8.5c+v+ef", cmap=True, frame=["a5", "+lNumber of independent GNSS velocity solutions"])
    
    # Save the figure
    fig.savefig("./results/figures/number_of_estimates.pdf", dpi=300)

    # Show the figure
    fig.show()

# Call the function
input_file = 'results/combined_velocities/statistics/site_statistics.csv'
plot_number_of_estimates(input_file)

---
## Step 8: Manual filtering of outliers

- Here I implement an approach that involves filtering GNSS stations given a list of coordinates and radii.
- The primary objective is removing stations affected by volcanic transients and outliers not removed in the previous filtering steps.


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from concurrent.futures import ThreadPoolExecutor
import glob

# Define the paths
temp_combined_folder_path = './results/combined_velocities/'
combined_folder_path = './results/combined_velocities/manual_filter/'

# Ignore future warnings (I will fix these in a future release)
warnings.simplefilter(action='ignore', category=FutureWarning) 

# Function to create necessary directories
def create_directory_for_file(file_path):
    directory = os.path.dirname(file_path)
    if not os.path.exists(directory):
        os.makedirs(directory, exist_ok=True)

# Haversine function
def haversine_distance(lon1, lat1, lon2, lat2):
    # Convert latitude and longitude from degrees to radians
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    radius = 6371  # Earth's radius in kilometers
    distance = radius * c

    return distance

def process_file(file_name):
    velocity_data_path = os.path.join(temp_combined_folder_path, file_name)
    filtered_data_path = os.path.join(combined_folder_path, file_name.replace(".csv", "_clean.csv"))
    log_path = os.path.join(combined_folder_path, file_name.replace(".csv", "_removed.log"))

    create_directory_for_file(filtered_data_path)
    create_directory_for_file(log_path)

    # Load GNSS velocity field data, specifying space as a separator
    velocity_data = pd.read_csv(velocity_data_path, sep=' ')
    original_data_count = len(velocity_data)  # Count of the original dataset

    # Load filter criteria, specifying space as a separator
    filter_criteria = pd.read_csv(filter_criteria_path, sep=' ')

    # Prepare log file
    log_data = pd.DataFrame(columns=velocity_data.columns)

    for _, criteria in filter_criteria.iterrows():
        center_lon, center_lat, radius = criteria['center_lon'], criteria['center_lat'], criteria['radius']
        
        # Calculate distances for all stations
        distances = haversine_distance(velocity_data['Lon'].values, velocity_data['Lat'].values,
                                       center_lon, center_lat)
        
        # Filter to remove stations inside the radius
        filter_mask = distances <= radius
        filtered_stations = velocity_data[filter_mask]
        velocity_data = velocity_data[~filter_mask]

        # Align columns of filtered_stations with log_data and drop all-NA columns
        filtered_stations_aligned = filtered_stations.reindex(columns=log_data.columns)
        filtered_stations_aligned.dropna(axis=1, how='all', inplace=True)

        # Append aligned and cleaned filtered stations to log
        log_data = pd.concat([log_data, filtered_stations_aligned], ignore_index=True)

        # Print debugging information only for 'eura' reference frame
        if 'eura' in file_name:
            print(f"Coordinate ({center_lon}, {center_lat}), Radius: {radius} km")
            print(f"Number of stations removed using this criterion: {filtered_stations.shape[0]}")

    total_removed_stations = len(log_data)  # Total number of stations removed
    percentage_removed = (total_removed_stations / original_data_count) * 100

    # Save the filtered data as space-separated
    velocity_data.to_csv(filtered_data_path, sep=' ', index=False)

    # Save the log data as space-separated
    log_data.to_csv(log_path, sep=' ', index=False)

    # Print summary information only for 'eura' reference frame
    if 'eura' in file_name:
        #print(f"Processed {file_name}")
        print("--------------------------------------------------------------")
        print(f"Total number of stations removed: {total_removed_stations}/{original_data_count}")
        print(f"Percentage of stations removed: {percentage_removed:.2f}%")
    
    # Return the path of the cleaned file
    return filtered_data_path

# Main execution
if __name__ == "__main__":
    print("--------------------------------------------------------------")
    print("            Filtering GNSS velocity fields manually            ")
    print("--------------------------------------------------------------")
    print()

    filter_criteria_path = './manual_filter/filter_criteria.csv'  

    # Get a list of all CSV files in the directory
    files = glob.glob(os.path.join(temp_combined_folder_path, 'combined_vel_*.csv'))

    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_file, os.path.basename(file)) for file in files]

    cleaned_files = [future.result() for future in futures]

    print("--------------------------------------------------------------")
    print("                   All files processed                        ")
    print("--------------------------------------------------------------")

    # Print the paths to the cleaned velocity CSV files
    print("       Check final velocity fields in the paths below:        ")
    print("--------------------------------------------------------------")
    for file_path in cleaned_files:
        print(file_path)
    print("--------------------------------------------------------------")

---
### Plot combined velocity field in different reference frames

In [ ]:
# User flag: set to True to enable plotting
do_plots = False  # Change to True to enable plotting

if do_plots:
    # Plotting rotated velocities before cleaning
    print("---------------------------------------------------")
    print("      Plotting final combined velocity fields      ")
    print("---------------------------------------------------")
    print()

    subprocess.run(["python", plot_rotated_script_path, combined_folder_path])

    print()
    print("--------------------------------------------------------------")
    print("Plotting combined velocity fields in different reference frames completed")
    print()
else:
    print("Plotting skipped because do_plots is False.")

#### Final results: Clean combined horizontal velocity fields in different reference frames

In [ ]:
"""Plotting horizontal velocity fields in different reference frames"""

import sys, os
sys.path.append(os.path.abspath("./scripts"))
from plot_gps_velocity_fields import plot_gps_velocity_fields

input_folder = "./results/combined_velocities/manual_filter/"

# Plot all reference frames:
#plot_gps_velocity_fields(input_folder)

# To plot a single plate (e.g. Eurasia-fixed only), use:
plot_gps_velocity_fields(input_folder, plate_name="eura")


#### **Combine vertical GNSS velocity fields**

- Here, I simply compute the median vertical velocity for each station considering a ridius of 1.2 km

In [ ]:
import os
import pandas as pd
import shutil

# Define the list of input files
input_files = [
    './results/input_files_rotation/serpelloni_2022.vel',
    './results/input_files_rotation/viltres_2022.vel',
#    './results/input_files_rotation/ergintav_2023.vel', Removed because of unreliable vertical uncertainties
    './results/input_files_rotation/euref_all.vel',
    './results/input_files_rotation/floyd_2023.vel',
    './results/input_files_rotation/euref_ch8.vel',
    './results/input_files_rotation/sokhadze_2018.vel',
    './results/input_files_rotation/castro_2021.vel',
    './results/input_files_rotation/briole_2021.vel', 
    './results/input_files_rotation/liang_2013.vel', 
    './results/input_files_rotation/pinaValdes_2022.vel',
    './results/input_files_rotation/jolivet_2023.vel'
]

# Define the destination folder
destination_folder = './results/input_verticals/'

# List of base filenames to modify U.vel and U.sig (assign NA to 0.00 values so that they are not considered in further processing)
# This is useful when some but not all GNSS sites have vertical velocities reported, such as in the case of Liang et al., 2013
modify_files = ['liang_2013']  # Here, users can add more base filenames as needed

# Check if the destination folder exists, and create it if it does not
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

# Function to read a .vel file and save the required columns to a new file
def process_vel_file(file_path):
    if not os.path.exists(file_path):
        print(f"File does not exist: {file_path}")
        return

    # Read the .vel file
    col_names = ['Lon', 'Lat', 'E.vel', 'N.vel', 'E.adj', 'N.adj', 'E.sig', 'N.sig', 'Corr', 'U.vel', 'U.adj', 'U.sig', 'Stat']
    data = pd.read_csv(file_path, sep=r'\s+', header=None, names=col_names)
    
    # Select the required columns
    selected_data = data[['Lon', 'Lat', 'U.vel', 'U.sig', 'Stat']].copy()
    
    # Check if the base filename is in the modify list
    base_filename = os.path.basename(file_path).split('.')[0]
    if base_filename in modify_files:
        selected_data.loc[selected_data['U.vel'] == 0.00, 'U.vel'] = pd.NA
        selected_data.loc[selected_data['U.sig'] == 0.00, 'U.sig'] = pd.NA
    
    # Check if there are any negative values in the U.sig column
    if (selected_data['U.sig'] < 0).any():
        print(f"Warning: Negative U.sig values found in U.sig column for {base_filename}.vel. Replacing with absolute values...")
        # Replace negative values with their absolute values
        selected_data['U.sig'] = selected_data['U.sig'].abs()
    
    # Save the selected columns to the destination folder with the same file name
    output_file_path = os.path.join(destination_folder, os.path.basename(file_path))
    selected_data.to_csv(output_file_path, sep=' ', index=False, header=False)

# Process each input file
for file_path in input_files:
    process_vel_file(file_path)

# Copy the .raw files from the source folder to the destination folder with a .vel extension
raw_input_folder = './raw_input/levelling_verticals/'
if os.path.exists(raw_input_folder):
    raw_files = [f for f in os.listdir(raw_input_folder) if f.endswith('.raw')]
    for raw_file in raw_files:
        src_path = os.path.join(raw_input_folder, raw_file)
        dst_path = os.path.join(destination_folder, raw_file.replace('.raw', '.vel'))
        shutil.copy(src_path, dst_path)
else:
    print(f"Folder does not exist: {raw_input_folder}")

# Check the result
print(f"\nFiles with vertical velocities copied to {destination_folder} folder:")
for file in os.listdir(destination_folder):
    print(file)


In [ ]:
import sys, os
sys.path.append(os.path.abspath("./scripts"))
import combine_verticals as cv

destination_folder = "./results/input_verticals/"
output_combined_verticals = "./results/combined_velocities/manual_filter"
log_file_path = "./results/combined_velocities/manual_filter/debug_combined_vertical_velocity_field.log"

output_file_path, _ = cv.combine_verticals(
    input_folder=destination_folder,
    output_folder=output_combined_verticals,
    log_file_path=log_file_path,
)

print(f"Combined vertical velocity field saved to: {output_file_path}")
print(f"Debug log saved to: {log_file_path}")


In [ ]:
# Histogram showing the distribution of vertical velocities 

import pandas as pd
import matplotlib.pyplot as plt

# Load the combined vertical velocity field
combined_vertical_file = './results/combined_velocities/manual_filter/combined_vertical_velocity_field.vel'
col_names = ['Lon', 'Lat', 'U.vel', 'U.sig', 'Stat']
combined_data = pd.read_csv(combined_vertical_file, sep=r'\s+', header=None, names=col_names)

# Define the threshold for removing vertical velocities
threshold = 2 # Number of standard deviations

# Plot histogram of vertical velocities
plt.figure(figsize=(10, 6))
plt.hist(combined_data['U.vel'], bins=150, color='lightgray', edgecolor='black')
plt.xlabel('Vertical velocity (mm/yr)')
plt.ylabel('Number of GNSS stations')
plt.tight_layout()
plt.xlim(-15, 15) # axis from -50 to 50 mm/yr
# comptute max number of stations in a bin for y-axis
max_freq = max(plt.hist(combined_data['U.vel'], bins=150, color='lightgray', edgecolor='black')[0])
plt.ylim(0, max_freq + 500)
# compute mean and standard deviation
mean = combined_data['U.vel'].mean()
std = combined_data['U.vel'].std()
plt.axvline(mean, color='b', linestyle='dashed', linewidth=1, label='Mean')
plt.axvline(mean + threshold*std, color='r', linestyle='dashed', linewidth=1, label=f'Mean $\pm$ {threshold}$\sigma$')
plt.axvline(mean - threshold*std, color='r', linestyle='dashed', linewidth=1)
plt.legend()
plt.savefig('./results/figures/vertical_velocity_histogram.pdf')
plt.show()

# Remove vertical velocities with magnitudes exceeding "threshold" standard deviations from the mean

# Compute the mean and standard deviation of the vertical velocities
mean = combined_data['U.vel'].mean()
std = combined_data['U.vel'].std()

# Remove vertical velocities with magnitudes exceeding the threshold
combined_data_filtered = combined_data[
    (combined_data['U.vel'] >= mean - threshold * std) &
    (combined_data['U.vel'] <= mean + threshold * std)
]

# Save the filtered vertical velocities to a new file
filtered_file_path = './results/combined_velocities/manual_filter/combined_vertical_velocity_field_mag_filtered.vel'
combined_data_filtered.to_csv(filtered_file_path, sep=' ', index=False, header=False)

# Print the result
print(f"Filtered vertical velocities saved to: {filtered_file_path}")


In [ ]:
# Import all functions from the module
import sys
import os

# Add the scripts folder to the Python path
sys.path.append(os.path.abspath('./scripts'))

from uncertainty_filter_verticals import UncertaintyFilterVerticals

# Define the file path and output locations
input_file = './results/combined_velocities/manual_filter/combined_vertical_velocity_field_mag_filtered.vel'
output_folder = './results/combined_velocities/manual_filter/'
figures_path = './results/figures/'

# Create an instance of the UncertaintyFilterVerticals class
filter_verticals = UncertaintyFilterVerticals(input_file, output_folder, figures_path)

# Read the vertical velocities
filter_verticals.read_vertical_velocities()

# Plot the uncertainty distribution
filter_verticals.plot_uncertainty_distribution()

# Filter the uncertainties and save the filtered velocities
filter_verticals.filter_uncertainties()

In [ ]:
import pandas as pd


# Here I count number of stations in the combined vertical velocity field within longitude and latitude ranges: 
# [5, 60] and [-20, 125], respectively. 

# Load the combined vertical velocity field
combined_vertical_file = './results/combined_velocities/manual_filter/filtered_vertical_velocity_field.vel'
col_names = ['Lon', 'Lat', 'U.vel', 'U.sig', 'Stat']
combined_data = pd.read_csv(combined_vertical_file, sep=r'\s+', header=None, names=col_names)

# Count the number of stations within the specified longitude and latitude ranges (Alpine-Himalayan Belt region)
lon_mask = (combined_data['Lon'] >= 5) & (combined_data['Lon'] <= 60)
lat_mask = (combined_data['Lat'] >= -20) & (combined_data['Lat'] <= 125)
stations_count = len(combined_data[lon_mask & lat_mask])

# Print the result
print(f"Number vertical velocities along the Alpine-Himalayan Belt region: {stations_count}")

In [ ]:
""" Plotting the vertical GNSS velocity field """

import pygmt
import pandas as pd

def plot_vertical_velocity_fields(input_files):

    # Create a new figure
    fig = pygmt.Figure()

    # Set the region and projection of the map
    fig.basemap(region=[-20, 125, 5, 60], projection='M20c', frame='af') # The whole Alpine-Himalayan belt

    # Create a custom color palette for the relief shading
    pygmt.makecpt(cmap="gray95,gray90,gray85", series=[-10000, 10000, 100])

    # Add shaded topography with 30% transparency
    fig.grdimage(grid="@earth_relief_03m", cmap=True, shading=True, transparency=20) # nan_transparent=True results in error in the latest GMT version

    # Add coastlines
    fig.coast(water='white', borders="1/0.1p,gray90", shorelines="0.1p,black", area_thresh=4000, resolution='h')

    # Create the custom colormap
    pygmt.makecpt(cmap='polar', series=[-4, 4, 0.01], continuous=True, background=True,)

    for input_file in input_files:

        # Read the CSV file
        df = pd.read_csv(input_file, sep=r'\s+', skiprows=1, header=None)
        df.columns = ['Lon', 'Lat', 'U.vel', 'U.sig', 'Stat']
   
        # Create circles data with fixed size and colors based on u_vel
        circles = []
        for lon, lat, norm_u in zip(df['Lon'], df['Lat'], df['U.vel']):
            circles.append([lon, lat, norm_u, 0.07])

        # Plot circles with color based on u_vel and fixed size
        fig.plot(
            data=circles,
            style='cc',
            cmap=True,
            #pen='0.01p,black',
        )
    
    # Add a colorbar to the plot
    fig.colorbar(position="JMR+o0.5c/0c+w9c+v+e", cmap=True, frame=["a0.1", "+lVertical velocity (mm/yr)"])

    # Add scale bar
    with pygmt.config(FONT_ANNOT_PRIMARY='8p', FONT_LABEL='8p'):
        fig.basemap(map_scale="JBR+o-9c/-0.8c+c0+w1000k+f+lkm")

    # Save the figure
    fig.savefig("./results/figures/vertical_velocity_fields.pdf", dpi=300) 

    # Show the figure
    fig.show()

# List of input files
input_files = [
    combined_vertical_file
]

# Call the function with the list of files
plot_vertical_velocity_fields(input_files)

## Steps 9 and 10: Scaling of horizontal GNSS velocity uncertainties and final filtering
- Here I follow the approach described in Piña-Valdez et al., (2022) to scale the distribution of velocity uncertainties in the combined velocity field, taking Wang & Barbot (2023) as the reference/target log-normal distribution of velocity uncertainties.
- Finally, I apply another filtering step based on the lognormal distribution of velocity uncertainties, removing sites with uncertainties exceeding the 99th percentile of the fitted log-normal distribution.

In [ ]:
# Import all functions from the module
import sys
import os

# Add the scripts folder to the Python path
sys.path.append(os.path.abspath('./scripts'))

import uncertainty_scaling_combined as uncertainty_scaling

# Set paths and run the harmonisation process
input_folder = './results/combined_velocities/manual_filter/' # Folder containing the combined velocity fields
reference_filename = './results/output_coherence_analysis/wang_barbot_2023.csv' # Reference file for uncertainty scaling
output_folder_scaled_results = './results/combined_velocities_scaled_uncertainties/'  # Output folder for results with scaled uncertainties

# Run the harmonisation/scaling process
uncertainty_scaling.harmonise_uncertainties(input_folder, reference_filename, output_folder_scaled_results)


### Plotting final velocities after uncertainty scaling and filtering:

In [ ]:
# Plotting rotated velocities before cleaning
print("---------------------------------------------------")
print("      Plotting final combined velocity fields      ")
print("---------------------------------------------------")
print()

subprocess.run(["python", plot_rotated_script_path, output_folder_scaled_results])

print()
print("--------------------------------------------------------------")
print("Plotting combined velocity fields in different reference frames completed")
print()